# Analisi di firme spettrali simulate

La **spettroscopia** è una tecnica fondamentale in astrofisica che analizza la luce emessa, assorbita o riflessa dai materiali. Ogni sostanza chimica ha una **firma spettrale** unica, determinata dalla sua struttura molecolare e atomica.

In questo notebook simuleremo le firme spettrali di diverse sostanze di interesse:
1. **Acqua (H₂O)**: assorbimento nell'infrarosso, cruciale per la ricerca di vita
2. **Clorofilla**: picco nel verde (fotosintesi, biomarcatore)
3. **Ossido di ferro (Fe₂O₃)**: caratteristico di superfici rocciose ossidate (Marte)
4. **Metano (CH₄)**: gas serra, presente su Titano e Giove

I dati sono **simulati** con profili gaussiani per scopi didattici.

## 1. Import librerie

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, Tuple, Optional

%matplotlib inline

## 2. Funzioni per la generazione di firme spettrali

In [ ]:
def profilo_gaussiano(wavelengths: np.ndarray, centro: float,
                      ampiezza: float, intensita: float = 1.0) -> np.ndarray:
    """Genera un profilo gaussiano (firma spettrale semplice).

    Args:
        wavelengths: Array delle lunghezze d'onda (nm)
        centro: Centro della gaussiana (nm)
        ampiezza: Deviazione standard della gaussiana (larghezza)
        intensita: Intensità massima del picco

    Returns:
        Array con il profilo gaussiano
    """
    return intensita * np.exp(-((wavelengths - centro) ** 2) / (2 * (ampiezza ** 2)))


def genera_firme_spettrali(wavelengths: np.ndarray) -> Dict[str, np.ndarray]:
    """Genera firme spettrali simulate per vari materiali.

    Ogni materiale è caratterizzato da uno o più picchi di
    assorbimento/emissione a specifiche lunghezze d'onda.

    Args:
        wavelengths: Array delle lunghezze d'onda (nm)

    Returns:
        Dizionario {nome_materiale: array_intensità}
    """
    firme = {}

    # Acqua: picco di assorbimento nell'infrarosso vicino
    firme['Acqua (H₂O)'] = (
        profilo_gaussiano(wavelengths, 600, 50, 1.0) +
        profilo_gaussiano(wavelengths, 750, 30, 0.6) +
        profilo_gaussiano(wavelengths, 450, 20, 0.3)
    )

    # Clorofilla: riflette nel verde, assorbe nel rosso e blu
    firme['Clorofilla'] = (
        profilo_gaussiano(wavelengths, 550, 40, 1.0) +
        profilo_gaussiano(wavelengths, 680, 25, 0.2) +
        profilo_gaussiano(wavelengths, 450, 30, 0.15)
    )

    # Ossido di ferro: ampia banda nel rosso (responsabile del colore di Marte)
    firme['Ossido di Ferro (Fe₂O₃)'] = (
        profilo_gaussiano(wavelengths, 650, 80, 0.9) +
        profilo_gaussiano(wavelengths, 500, 40, 0.4) +
        profilo_gaussiano(wavelengths, 400, 25, 0.2)
    )

    # Metano: bande di assorbimento nell'infrarosso
    firme['Metano (CH₄)'] = (
        profilo_gaussiano(wavelengths, 720, 20, 0.7) +
        profilo_gaussiano(wavelengths, 790, 25, 0.8) +
        profilo_gaussiano(wavelengths, 620, 15, 0.3)
    )

    return firme


# Generazione delle lunghezze d'onda (spettro visibile + infrarosso vicino)
wavelengths = np.linspace(300, 800, 500)
firme = genera_firme_spettrali(wavelengths)
print(f"Firme spettrali generate per {len(firme)} materiali.")
for nome in firme:
    print(f"  - {nome}: {len(firme[nome])} campioni")

## 3. Creazione di un DataFrame con i dati spettrali

In [ ]:
def crea_dataframe_spettrale(wavelengths: np.ndarray,
                              firme: Dict[str, np.ndarray]) -> pd.DataFrame:
    """Crea un DataFrame pandas con le firme spettrali.

    Args:
        wavelengths: Array delle lunghezze d'onda
        firme: Dizionario {nome: array_intensità}

    Returns:
        DataFrame con colonne ['Wavelength', nome1, nome2, ...]
    """
    df = pd.DataFrame({'Wavelength (nm)': wavelengths})
    for nome, intensita in firme.items():
        df[nome] = intensita
    return df


df_spettrale = crea_dataframe_spettrale(wavelengths, firme)
print("Prime 5 righe del DataFrame spettrale:")
display(df_spettrale.head())

## 4. Visualizzazione delle singole firme spettrali

In [ ]:
def plot_firma_spettrale(wavelengths: np.ndarray, intensita: np.ndarray,
                          titolo: str, colore: str = 'steelblue'):
    """Crea un grafico di una singola firma spettrale.

    Args:
        wavelengths: Array delle lunghezze d'onda (nm)
        intensita: Array delle intensità
        titolo: Titolo del grafico
        colore: Colore della linea
    """
    plt.figure(figsize=(10, 4))
    plt.plot(wavelengths, intensita, color=colore, linewidth=2)
    plt.title(titolo, fontsize=14, fontweight='bold')
    plt.xlabel('Lunghezza d\'onda (nm)', fontsize=12)
    plt.ylabel('Intensità (normalizzata)', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.ylim(-0.05, 1.1)

    # Evidenzia il massimo
    idx_max = np.argmax(intensita)
    plt.axvline(x=wavelengths[idx_max], color='crimson',
                linestyle='--', alpha=0.7,
                label=f'Picco a {wavelengths[idx_max]:.0f} nm')
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_firma_spettrale(wavelengths, firme['Acqua (H₂O)'],
                      'Firma spettrale: Acqua (H₂O)', 'steelblue')
plot_firma_spettrale(wavelengths, firme['Clorofilla'],
                      'Firma spettrale: Clorofilla', 'forestgreen')
plot_firma_spettrale(wavelengths, firme['Ossido di Ferro (Fe₂O₃)'],
                      'Firma spettrale: Ossido di Ferro', 'saddlebrown')
plot_firma_spettrale(wavelengths, firme['Metano (CH₄)'],
                      'Firma spettrale: Metano (CH₄)', 'darkorange')

## 5. Confronto di tutte le firme spettrali

Sovrapponiamo tutte le firme spettrali per confrontare i diversi profili. Ogni materiale ha una **impronta digitale** spettrale unica che permette di identificarlo a distanza tramite spettroscopia remota.

In [ ]:
def plot_confronto_firme(wavelengths: np.ndarray,
                          firme: Dict[str, np.ndarray]):
    """Confronta tutte le firme spettrali in un unico grafico.

    Args:
        wavelengths: Array delle lunghezze d'onda
        firme: Dizionario {nome: array_intensità}
    """
    colori = ['steelblue', 'forestgreen', 'saddlebrown', 'darkorange']

    plt.figure(figsize=(12, 6))

    for i, (nome, intensita) in enumerate(firme.items()):
        colore = colori[i % len(colori)]
        plt.plot(wavelengths, intensita, color=colore,
                 linewidth=2, label=nome, alpha=0.85)

    plt.title('Confronto di Firme Spettrali Simulate', fontsize=14, fontweight='bold')
    plt.xlabel('Lunghezza d\'onda (nm)', fontsize=12)
    plt.ylabel('Intensità (normalizzata)', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.ylim(-0.05, 1.2)
    plt.legend(fontsize=10, loc='upper right')

    # Evidenzia le regioni spettrali principali
    plt.axvspan(380, 450, alpha=0.08, color='blue', label='Blu')
    plt.axvspan(450, 495, alpha=0.08, color='green', label='Verde')
    plt.axvspan(495, 570, alpha=0.08, color='yellow', label='Giallo')
    plt.axvspan(570, 590, alpha=0.08, color='orange', label='Arancio')
    plt.axvspan(590, 780, alpha=0.08, color='red', label='Rosso')

    plt.tight_layout()
    plt.show()

plot_confronto_firme(wavelengths, firme)

## 6. Statistiche comparative

In [ ]:
def statistiche_spettrali(df: pd.DataFrame) -> pd.DataFrame:
    """Calcola statistiche di base per ogni firma spettrale.

    Args:
        df: DataFrame con colonne [Wavelength, firme...]

    Returns:
        DataFrame con statistiche per ogni materiale
    """
    colonne_firme = [c for c in df.columns if c != 'Wavelength (nm)']
    stats = {}

    for col in colonne_firme:
        dati = df[col]
        idx_max = dati.idxmax()
        stats[col] = {
            'Picco max': dati.max(),
            'λ al picco (nm)': df.loc[idx_max, 'Wavelength (nm)'],
            'Media': dati.mean(),
            'Dev. std': dati.std(),
            'Integrale': dati.sum()
        }

    df_stats = pd.DataFrame(stats).T
    df_stats.index.name = 'Materiale'
    return df_stats


df_stats = statistiche_spettrali(df_spettrale)
print("Statistiche delle firme spettrali:")
display(df_stats.round(4))

## Riepilogo

In questo notebook abbiamo:
- Simulato firme spettrali per 4 materiali (acqua, clorofilla, ossido di ferro, metano)
- Generato profili gaussiani con picchi caratteristici
- Creato un DataFrame pandas per l'analisi strutturata
- Visualizzato singolarmente e confrontato tutte le firme
- Calcolato statistiche comparative (picco, media, integrale)

La spettroscopia è uno strumento fondamentale per l'astrobiologia e la geologia planetaria: permette di identificare la composizione chimica di pianeti, stelle e galassie senza contatto diretto, analizzando la luce che emettono o riflettono.